In [43]:
"""
complexity_analysis.py
=======================
Computational complexity analysis for all sensor anomaly detection models.

Covers:
  1. Learnable parameter count  (per layer and total)
  2. FLOPs per forward pass      (via thop; analytical fallback per layer type)
  3. Theoretical complexity      (Big-O in N, T, H)
  4. Forward-pass latency        (CPU wall-clock; GPU if available)
  5. Peak memory usage           (CPU RSS; GPU VRAM if available)
  6. Throughput                  (samples / second)
  7. Scaling analysis            (how each metric grows as N and T increase)

Usage (Jupyter):
    from complexity_analysis import run_complexity_analysis
    run_complexity_analysis(num_sensors=10, seq_len=24, adj_csv="adjacency.csv")

Usage (CLI):
    python complexity_analysis.py --adj_csv adjacency.csv --num_sensors 10 --seq_len 24

Dependencies:
    pip install thop tabulate psutil torch torch-geometric pandas numpy
"""

import argparse
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# '' Optional imports ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
try:
    from thop import profile as thop_profile
    THOP = True
except ImportError:
    THOP = False
    print("[!]  thop not installed - FLOPs will use analytical estimates.\n"
          "   Install with: pip install thop")

try:
    import psutil
    PSUTIL = True
except ImportError:
    PSUTIL = False

try:
    from tabulate import tabulate
    TABULATE = True
except ImportError:
    TABULATE = False

try:
    from torch_geometric.nn import GATConv
    from torch_geometric.nn.dense import DenseGCNConv
    GNN_AVAILABLE = True
except ImportError:
    GNN_AVAILABLE = False
    print("[!]  torch_geometric not found - GNN models will be skipped.")


# =============================================================================
# SECTION 1 - MODEL DEFINITIONS
# (Minimal copies of models from anomaly_pipeline.py, kept self-contained)
# =============================================================================

# '' CNN '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1, **kw):
        pad = (kernel_size - 1) * dilation
        super().__init__(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation, **kw)
        self._pad = pad

    def forward(self, x):
        out = super().forward(x)
        return out[:, :, :-self._pad] if self._pad > 0 else out


class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.c1  = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.c2  = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.n1  = nn.BatchNorm1d(out_ch)
        self.n2  = nn.BatchNorm1d(out_ch)
        self.res = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.drop = nn.Dropout(0.1)

    def forward(self, x):
        out = self.drop(torch.relu(self.n1(self.c1(x))))
        out = self.drop(torch.relu(self.n2(self.c2(out))))
        return torch.relu(out + self.res(x))


class TemporalCNN(nn.Module):
    """
    Temporal CNN (TCN): stacked dilated causal convolutions.
    Input  : (B, T, N)
    Output : (B,)  ' binary logit
    """
    def __init__(self, num_sensors, seq_len, hidden_dim=32, num_levels=4, kernel_size=3):
        super().__init__()
        chs = [num_sensors] + [hidden_dim] * num_levels
        self.net = nn.Sequential(*[
            TCNBlock(chs[i], chs[i+1], kernel_size, 2**i)
            for i in range(num_levels)
        ])
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out = self.net(x.permute(0, 2, 1))[:, :, -1]
        return self.fc(out).squeeze(-1)


# '' LSTM ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
class StackedLSTM(nn.Module):
    """
    Stacked LSTM encoder + linear decoder.
    Input  : (B, T, N)
    Output : (B,)
    """
    def __init__(self, num_sensors, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(num_sensors, hidden_dim, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :])).squeeze(-1)


# '' VAE '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
class SensorVAE(nn.Module):
    """
    LSTM-based VAE.
    Input  : (B, T, N)
    Output : recon (B, T, N), mu (B, latent), logvar (B, latent)
    """
    def __init__(self, num_sensors, seq_len, hidden_dim=64, latent_dim=16, beta=1.0):
        super().__init__()
        self.beta    = beta
        self.seq_len = seq_len
        # Encoder
        self.enc_lstm   = nn.LSTM(num_sensors, hidden_dim, batch_first=True)
        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar  = nn.Linear(hidden_dim, latent_dim)
        # Decoder
        self.dec_fc     = nn.Linear(latent_dim, hidden_dim)
        self.dec_lstm   = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.dec_out    = nn.Linear(hidden_dim, num_sensors)

    def forward(self, x):
        _, (h, _) = self.enc_lstm(x)
        h  = h[-1]
        mu = self.fc_mu(h)
        lv = self.fc_logvar(h)
        z  = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        d  = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        o, _ = self.dec_lstm(d)
        return self.dec_out(o), mu, lv

    def anomaly_score(self, x):
        self.eval()
        with torch.no_grad():
            recon, _, _ = self.forward(x)
        return F.mse_loss(recon, x, reduction='none').mean(dim=(1, 2))


# '' GNN models ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
if GNN_AVAILABLE:

    class SpatioTemporalGNN2(nn.Module):
        """Fixed-graph GCN + GRU (precomputed normalised adjacency)."""
        def __init__(self, num_sensors, edge_index, gcn_hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors    = num_sensors
            self.gcn_hidden_dim = gcn_hidden_dim
            A     = torch.zeros(num_sensors, num_sensors)
            A[edge_index[0], edge_index[1]] = 1.0
            A     = A + torch.eye(num_sensors)
            deg   = A.sum(dim=1).clamp(min=1e-6)
            d_inv = deg.pow(-0.5)
            self.register_buffer("A_hat", d_inv.unsqueeze(1) * A * d_inv.unsqueeze(0))
            self.W        = nn.Linear(1, gcn_hidden_dim, bias=False)
            self.gcn_bias = nn.Parameter(torch.zeros(gcn_hidden_dim))
            self.rnn      = nn.GRU(gcn_hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc       = nn.Linear(rnn_hidden_dim, 1)

        def forward(self, x, edge_index=None):
            B, T, N = x.shape
            node_feats = self.W(x.unsqueeze(-1))
            gcn_out    = torch.relu(
                torch.einsum("ij,btjh->btih", self.A_hat, node_feats) + self.gcn_bias
            )
            rnn_input = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)
            _, h_n    = self.rnn(rnn_input)
            rnn_out   = h_n.squeeze(0).reshape(B, N, -1)
            return self.fc(rnn_out.mean(dim=1)).squeeze(-1)

    class GNN_LearnableGraph(nn.Module):
        """Learnable adjacency (dense GCN) + GRU."""
        def __init__(self, num_sensors, hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors = num_sensors
            self.adj_logits  = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gcn_layer   = DenseGCNConv(in_channels=1, out_channels=hidden_dim)
            self.rnn         = nn.GRU(hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc          = nn.Linear(rnn_hidden_dim, 1)

        def forward(self, x):
            B, T, N = x.shape
            adj       = torch.softmax(self.adj_logits, dim=1)
            adj       = adj * (1 - torch.eye(N, device=x.device))
            adj_batch = adj.unsqueeze(0).expand(B, -1, -1)
            gcn_outs  = [self.gcn_layer(x[:, t, :].unsqueeze(-1), adj_batch)
                         for t in range(T)]
            gcn_out   = torch.stack(gcn_outs, dim=1)
            rnn_input = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)
            _, h_n    = self.rnn(rnn_input)
            rnn_out   = h_n.squeeze(0).reshape(B, N, -1)
            return self.fc(rnn_out.mean(dim=1)).squeeze(-1)

    class GATLSTM_LearnableGraph(nn.Module):
        """
        GAT + LSTM with LEARNED edge structure (fully-connected candidate graph).
        The graph is an N'N learnable mask: every possible edge exists, but
        its weight is a sigmoid-gated learnable parameter. GAT attention then
        further refines edge weights per head.
        """
        def __init__(self, num_sensors, seq_len, hidden_dim=16, heads=2):
            super().__init__()
            self.num_sensors = num_sensors
            self.edge_logits = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gat  = GATConv(seq_len, hidden_dim, heads=heads,
                                concat=True, dropout=0.1)
            self.lstm = nn.LSTM(num_sensors * hidden_dim * heads,
                                hidden_dim, batch_first=True)
            self.fc   = nn.Linear(hidden_dim, 1)
            self.relu = nn.ReLU()
            self.drop = nn.Dropout(0.1)

        def _get_edge_index(self, device):
            # Gate edges with a learnable mask (top-k selection each forward pass).
            # Keep all edges above sigmoid threshold 0.5.
            mask = torch.sigmoid(self.edge_logits) > 0.5
            mask.fill_diagonal_(False)
            src, dst = torch.where(mask)
            if len(src) == 0:   # fallback: dense graph if all gated off
                src, dst = torch.where(~torch.eye(self.num_sensors, dtype=torch.bool, device=device))
            return torch.stack([src, dst], dim=0).to(device)

        def forward(self, x):
            B = x.shape[0]
            edge_index = self._get_edge_index(x.device)
            gat_outs = [self.drop(self.relu(self.gat(x[i].T, edge_index)))
                        for i in range(B)]
            gat_out  = torch.stack(gat_outs).reshape(B, 1, -1)
            out, _   = self.lstm(gat_out)
            return self.fc(out[:, -1, :]).squeeze(-1)

    class GCNForecaster_Learnable(nn.Module):
        """
        Unsupervised GCN forecaster with LEARNED adjacency matrix.
        Identical architecture to SpatioTemporalGNN2 but with adj_logits
        (N'N learnable params) instead of a precomputed buffer, and
        predicts the full N-dim next step instead of a binary logit.
        """
        def __init__(self, num_sensors, gcn_hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors    = num_sensors
            self.gcn_hidden_dim = gcn_hidden_dim
            self.adj_logits = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.W          = nn.Linear(1, gcn_hidden_dim, bias=False)
            self.gcn_bias   = nn.Parameter(torch.zeros(gcn_hidden_dim))
            self.rnn        = nn.GRU(gcn_hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc         = nn.Linear(rnn_hidden_dim, num_sensors)

        def _get_adj(self, device):
            adj = torch.softmax(self.adj_logits, dim=1)
            return adj * (1 - torch.eye(self.num_sensors, device=device))

        def forward(self, x):
            B, T, N = x.shape
            x_in       = x[:, :-1, :]
            adj        = self._get_adj(x.device)
            node_feats = self.W(x_in.unsqueeze(-1))
            gcn_out    = torch.relu(
                torch.einsum("ij,btjh->btih", adj, node_feats) + self.gcn_bias
            )
            rnn_in = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
            out, _ = self.rnn(rnn_in)
            last   = out[:, -1, :].reshape(B, N, -1).mean(dim=1)
            return self.fc(last)

    class LearnableGraphForecaster(nn.Module):
        """Unsupervised forecaster with learnable adjacency."""
        def __init__(self, num_sensors, hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors = num_sensors
            self.adj_logits  = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gcn_layer   = DenseGCNConv(in_channels=1, out_channels=hidden_dim)
            self.rnn         = nn.GRU(hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc          = nn.Linear(rnn_hidden_dim, num_sensors)

        def forward(self, x):
            B, T, N = x.shape
            x_in     = x[:, :-1, :]
            adj      = torch.softmax(self.adj_logits, dim=1)
            adj      = adj * (1 - torch.eye(N, device=x.device))
            adj_b    = adj.unsqueeze(0).expand(B, -1, -1)
            gcn_outs = [self.gcn_layer(x_in[:, t, :].unsqueeze(-1), adj_b)
                        for t in range(T - 1)]
            gcn_out  = torch.stack(gcn_outs, dim=1)
            rnn_in   = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
            out, _   = self.rnn(rnn_in)
            last     = out[:, -1, :].reshape(B, N, -1).mean(dim=1)
            return self.fc(last)

    class GATForecaster_Learnable(nn.Module):
        """
        Unsupervised GAT forecaster with LEARNED edge structure.
        Same gated-edge approach as GATLSTM_LearnableGraph.
        """
        def __init__(self, num_sensors, seq_len, hidden_dim=16, heads=2):
            super().__init__()
            self.num_sensors = num_sensors
            self.edge_logits = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gat  = GATConv(seq_len - 1, hidden_dim, heads=heads,
                                concat=True, dropout=0.1)
            self.lstm = nn.LSTM(num_sensors * hidden_dim * heads,
                                hidden_dim, batch_first=True)
            self.fc   = nn.Linear(hidden_dim, num_sensors)
            self.relu = nn.ReLU()
            self.drop = nn.Dropout(0.1)

        def _get_edge_index(self, device):
            mask = torch.sigmoid(self.edge_logits) > 0.5
            mask.fill_diagonal_(False)
            src, dst = torch.where(mask)
            if len(src) == 0:
                src, dst = torch.where(~torch.eye(self.num_sensors, dtype=torch.bool, device=device))
            return torch.stack([src, dst], dim=0).to(device)

        def forward(self, x):
            B, T, N = x.shape
            x_in = x[:, :-1, :]
            edge_index = self._get_edge_index(x.device)
            gat_outs = [self.drop(self.relu(self.gat(x_in[i].T, edge_index)))
                        for i in range(B)]
            gat_out  = torch.stack(gat_outs).reshape(B, 1, -1)
            out, _   = self.lstm(gat_out)
            return self.fc(out[:, -1, :])

    class GraphAnomalyVAE_Learnable(nn.Module):
        """
        Graph-aware VAE with LEARNED adjacency matrix.
        The adjacency is softmax-normalised from adj_logits (N'N trainable).
        """
        def __init__(self, num_sensors, seq_len,
                     gcn_hidden=16, rnn_hidden=32, latent_dim=16, beta=1.0):
            super().__init__()
            self.num_sensors = num_sensors
            self.seq_len     = seq_len
            self.beta        = beta
            self.adj_logits  = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.enc_node_proj = nn.Linear(1, gcn_hidden, bias=False)
            self.dec_node_proj = nn.Linear(1, gcn_hidden, bias=False)
            self.enc_rnn   = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
            self.fc_mu     = nn.Linear(rnn_hidden, latent_dim)
            self.fc_logvar = nn.Linear(rnn_hidden, latent_dim)
            self.dec_fc    = nn.Linear(latent_dim, rnn_hidden)
            self.dec_rnn   = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
            self.dec_gcn_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
            self.dec_out   = nn.Linear(gcn_hidden, 1)

        def _get_adj(self, device):
            adj = torch.softmax(self.adj_logits, dim=1)
            return adj * (1 - torch.eye(self.num_sensors, device=device))

        def forward(self, x):
            B, T, N = x.shape
            adj       = self._get_adj(x.device)
            feats     = self.enc_node_proj(x.unsqueeze(-1))
            gcn_out   = torch.relu(torch.einsum("ij,btjh->btih", adj, feats))
            rnn_input = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)
            _, h_n    = self.enc_rnn(rnn_input)
            h         = h_n.squeeze(0).reshape(B, N, -1).mean(dim=1)
            mu, lv    = self.fc_mu(h), self.fc_logvar(h)
            z         = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
            d         = self.dec_fc(z).unsqueeze(1).repeat(1, T, 1)
            rnn_out, _ = self.dec_rnn(d)
            node_h    = self.dec_gcn_W(rnn_out).unsqueeze(2).repeat(1, 1, N, 1)
            prop      = torch.relu(torch.einsum("ij,btjh->btih", adj, node_h))
            return self.dec_out(prop).squeeze(-1), mu, lv

        def anomaly_score(self, x):
            self.eval()
            with torch.no_grad():
                recon, _, _ = self.forward(x)
            return F.mse_loss(recon, x, reduction='none').mean(dim=(1, 2))


# =============================================================================
# SECTION 2 - ADJACENCY MATRIX LOADER
# =============================================================================

def load_edge_index(adj_csv_path: str, num_sensors: int):
    """Load adjacency matrix CSV and return (edge_index, adj_matrix)."""
    df  = pd.read_csv(adj_csv_path, index_col=None)
    first_col_numeric = pd.to_numeric(df.iloc[:, 0], errors="coerce").notna().all()
    if not first_col_numeric:
        df = pd.read_csv(adj_csv_path, index_col=0)
        adj = df.values.astype(np.float32)
    else:
        adj = df.values.astype(np.float32)

    if adj.shape[0] != num_sensors:
        raise ValueError(
            f"Adjacency matrix is {adj.shape[0]}'{adj.shape[0]} but "
            f"num_sensors={num_sensors}. Adjust --num_sensors."
        )
    np.fill_diagonal(adj, 0)
    rows, cols = np.where(adj > 0)
    edge_index = torch.tensor(np.stack([rows, cols], axis=0), dtype=torch.long)
    n_edges    = edge_index.shape[1]
    density    = n_edges / max(num_sensors * (num_sensors - 1), 1)
    print(f"   Adjacency: {num_sensors} sensors, {n_edges} directed edges, "
          f"density={100*density:.1f}%")
    return edge_index, adj


def make_synthetic_edge_index(num_sensors: int, num_neighbors: int = 3):
    """Fallback: build a random sparse edge_index when no CSV is provided."""
    rng = np.random.default_rng(42)
    rows, cols = [], []
    for i in range(num_sensors):
        nbrs = rng.choice(
            [j for j in range(num_sensors) if j != i],
            size=min(num_neighbors, num_sensors - 1),
            replace=False
        )
        for j in nbrs:
            rows += [i, j]; cols += [j, i]
    ei = torch.tensor([rows, cols], dtype=torch.long)
    return torch.unique(ei, dim=1)


# =============================================================================
# SECTION 3 - PARAMETER COUNT (per layer and total)
# =============================================================================

def count_parameters(model: nn.Module) -> dict:
    """
    Return a dict with per-layer and total learnable parameter counts.
    Buffers (e.g. A_hat) are counted separately as non-trainable.
    """
    layer_params  = {}
    total_train   = 0
    total_buffer  = 0

    for name, param in model.named_parameters():
        n = param.numel()
        layer_params[name] = n
        total_train += n

    for name, buf in model.named_buffers():
        total_buffer += buf.numel()

    return {
        "per_layer"     : layer_params,
        "total_trainable" : total_train,
        "total_buffer"  : total_buffer,   # e.g. precomputed A_hat
    }


def _fmt_params(n: int) -> str:
    if n >= 1_000_000: return f"{n/1e6:.3f} M"
    if n >= 1_000:     return f"{n/1e3:.1f} K"
    return str(n)


# =============================================================================
# SECTION 4 - FLOPs (thop + analytical per-layer fallback)
# =============================================================================

def _analytical_flops(model: nn.Module, dummy_inputs: tuple,
                       N: int, T: int, B: int = 1) -> str:
    """
    Layer-by-layer FLOPs estimate (MACs - 2 = FLOPs).

    Covers:
      nn.Linear / nn.Conv1d / nn.GRU / nn.LSTM  (exact)
      GCN propagation via  '... @ X        (N' ' H per time step)
      DenseGCNConv  via  adj @ X @ W    (N' ' H  +  N - H_in - H_out per step)
      GAT attention               (E - H ' heads)
      softmax over adj_logits     (N' per forward pass)

    The GCN/GAT estimates are based on model attribute names:
      - Modules containing 'gcn' or an  A_hat  buffer  '  GCN propagation cost
      - Modules containing 'gat' or a GATConv instance '  GAT edge attention cost
      - Parameters named 'adj_logits' or 'edge_logits' '  softmax / sigmoid cost

    Returns a human-readable string with '(est.)' suffix.
    """
    total_macs = 0
    has_graph_prop = False

    # '' Standard layer costs ''''''''''''''''''''''''''''''''''''''''''''''''''
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            total_macs += B * module.out_features * module.in_features
        elif isinstance(module, nn.Conv1d):
            L_out = T
            total_macs += (B * module.out_channels * L_out
                           * (module.in_channels // module.groups)
                           * module.kernel_size[0])
        elif isinstance(module, nn.GRU):
            H, I, L = module.hidden_size, module.input_size, module.num_layers
            total_macs += B * T * L * 3 * (H * H + H * I)
        elif isinstance(module, nn.LSTM):
            H, I, L = module.hidden_size, module.input_size, module.num_layers
            total_macs += B * T * L * 4 * (H * H + H * I)

    # '' Graph-specific costs (GCN propagation '... @ X via einsum) '''''''''''''''
    # Detect by presence of A_hat buffer or adj_logits / edge_logits parameter
    gcn_hidden = None
    for pname, _ in model.named_parameters():
        if "gcn" in pname.lower() or pname.endswith("W.weight"):
            # Try to infer hidden dim from model attributes
            for mod in model.modules():
                if isinstance(mod, nn.Linear) and mod.in_features == 1:
                    gcn_hidden = mod.out_features
                    break
            break

    # Detect GNN models: they have either A_hat buffer OR adj_logits parameter
    is_gnn = False
    for bname, _ in model.named_buffers():
        if "a_hat" in bname.lower():
            is_gnn = True
            break
    if not is_gnn:
        for pname, _ in model.named_parameters():
            if "adj_logits" in pname or "edge_logits" in pname:
                is_gnn = True
                break

    if is_gnn:
        # Estimate graph propagation cost.
        # '... @ X  where '... is (N,N) and X is (B, T, N, H) '  (B'T) matmuls of
        # size (N - N) ' (N - H) = N' ' H  per sample per time step.
        # Most models propagate over T steps (some over T-1); use T as estimate.
        if gcn_hidden is None:
            gcn_hidden = 16   # fallback based on our default config

        graph_prop_macs = B * T * N * N * gcn_hidden
        total_macs += graph_prop_macs

        # For decoder in GraphAnomalyVAE: a second graph propagation pass
        for mname, _ in model.named_modules():
            if "dec_rnn" in mname:
                total_macs += graph_prop_macs   # decoder also propagates
                break

        # Softmax over adj_logits (when learnable) contributes N' ops per forward
        for pname, _ in model.named_parameters():
            if "adj_logits" in pname or "edge_logits" in pname:
                total_macs += N * N * 3   # softmax: exp + sum + div
                break

        has_graph_prop = True

    # '' GATConv: attention over E edges '''''''''''''''''''''''''''''''''''''''
    if GNN_AVAILABLE:
        for mod in model.modules():
            if isinstance(mod, GATConv):
                H_in  = mod.in_channels
                H_out = mod.out_channels
                heads = mod.heads
                # Per sample: N nodes - H_in ' (H_out - heads) for linear transform
                # Plus ~E edges ' (H_out - heads) for attention
                # Approximate E as N' (dense) for the learnable-edge case
                E_approx = N * N
                total_macs += B * (N * H_in * H_out * heads
                                   + E_approx * H_out * heads)
                has_graph_prop = True

    flops = int(total_macs * 2)
    suffix = " (est.)"
    if not has_graph_prop:
        suffix = " (est., layers only)"

    if flops >= 1e9: return f"{flops/1e9:.3f} G{suffix}"
    if flops >= 1e6: return f"{flops/1e6:.2f} M{suffix}"
    if flops >= 1e3: return f"{flops/1e3:.1f} K{suffix}"
    return f"{flops}{suffix}"


def count_flops(model: nn.Module, dummy_inputs: tuple,
                N: int, T: int, B: int = 1) -> str:
    """
    Attempt thop profiling on CPU.
    Falls back to analytical estimate if thop fails or is unavailable.
    """
    if THOP:
        try:
            cpu_model  = model.to("cpu")
            cpu_inputs = tuple(i.to("cpu") if isinstance(i, torch.Tensor) else i
                               for i in dummy_inputs)
            macs, _ = thop_profile(cpu_model, inputs=cpu_inputs, verbose=False)
            flops = int(macs * 2)
            if flops >= 1e9: return f"{flops/1e9:.3f} G"
            if flops >= 1e6: return f"{flops/1e6:.2f} M"
            if flops >= 1e3: return f"{flops/1e3:.1f} K"
            return str(flops)
        except Exception:
            pass   # fall through to analytical

    return _analytical_flops(model, dummy_inputs, N, T, B)


# =============================================================================
# SECTION 5 - LATENCY & THROUGHPUT
# =============================================================================

def measure_latency_and_throughput(model: nn.Module, dummy_inputs: tuple,
                                   device: torch.device,
                                   n_warmup: int = 10,
                                   n_runs: int   = 50,
                                   batch_size: int = 1) -> dict:
    """
    Measure:
      latency_ms   : average single-forward-pass wall-clock time (ms)
      throughput   : samples per second
    """
    model = model.to(device).eval()
    inputs = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                   for i in dummy_inputs)

    with torch.no_grad():
        for _ in range(n_warmup):
            model(*inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(n_runs):
            model(*inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

    latency_ms = (elapsed / n_runs) * 1000
    throughput  = (batch_size * n_runs) / elapsed
    return {"latency_ms": round(latency_ms, 3),
            "throughput": round(throughput, 1)}


# =============================================================================
# SECTION 6 - MEMORY USAGE
# =============================================================================

def measure_memory(model: nn.Module, dummy_inputs: tuple,
                   device: torch.device) -> dict:
    """
    Measure peak memory for one forward pass.
      CPU: RSS delta via psutil (approximate)
      GPU: torch.cuda.max_memory_allocated
    """
    model = model.to(device).eval()
    inputs = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                   for i in dummy_inputs)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            model(*inputs)
        peak_mb = torch.cuda.max_memory_allocated(device) / 1024**2
        return {"device": "GPU", "peak_mb": round(peak_mb, 2)}

    elif PSUTIL:
        proc = psutil.Process()
        mem_before = proc.memory_info().rss / 1024**2
        with torch.no_grad():
            model(*inputs)
        mem_after = proc.memory_info().rss / 1024**2
        delta = max(mem_after - mem_before, 0.0)
        return {"device": "CPU", "peak_mb": round(delta, 2)}

    else:
        return {"device": "CPU", "peak_mb": "N/A (install psutil)"}


# =============================================================================
# SECTION 7 - THEORETICAL COMPLEXITY TABLE
# =============================================================================

THEORETICAL_COMPLEXITY = {
    "TemporalCNN": {
        "time"   : "O(T - C ' K - L)",
        "space"  : "O(T - C)",
        "notes"  : "C=hidden channels, K=kernel size, L=levels. "
                   "Fully parallel across time.",
        "graph_aware": False,
        "paradigm"   : "Supervised",
        "trainable_graph_params": 0,
    },
    "StackedLSTM": {
        "time"   : "O(T - H' ' L)",
        "space"  : "O(T - H)",
        "notes"  : "H=hidden dim, L=num_layers. Sequential time steps.",
        "graph_aware": False,
        "paradigm"   : "Supervised",
        "trainable_graph_params": 0,
    },
    "SensorVAE": {
        "time"   : "O(T - H') ' 2  (encoder + decoder)",
        "space"  : "O(T - H + latent_dim)",
        "notes"  : "LSTM encoder + LSTM decoder.",
        "graph_aware": False,
        "paradigm"   : "Unsupervised (ELBO)",
        "trainable_graph_params": 0,
    },
    "SpatioTemporalGNN2\n(ONLY fixed-graph model)": {
        "time"   : "O(T - N' ' H + T - N ' H')",
        "space"  : "O(N' buffer + T - N ' H)",
        "notes"  : "'... precomputed from edge_index, stored as non-trainable buffer. "
                   "No graph parameters to learn - graph structure is given.",
        "graph_aware": True,
        "paradigm"   : "Supervised",
        "trainable_graph_params": 0,
    },
    "GNN_LearnableGraph\n(learned adjacency)": {
        "time"   : "O(T - N' ' H + T - N ' H' + N')",
        "space"  : "O(N' param + T - N ' H)",
        "notes"  : "N' learnable adj_logits + softmax cost per forward pass.",
        "graph_aware": True,
        "paradigm"   : "Supervised",
        "trainable_graph_params": "N'",
    },
    "GATLSTM_LearnableGraph\n(learned edges)": {
        "time"   : "O(B - E ' H - heads + H')",
        "space"  : "O(N' param + E - H ' heads)",
        "notes"  : "N' edge_logits (sigmoid-gated) + GAT attention over active edges.",
        "graph_aware": True,
        "paradigm"   : "Supervised",
        "trainable_graph_params": "N'",
    },
    "GraphAnomalyVAE_Learnable": {
        "time"   : "O(T - N' ' H + T - N ' H') ' 2",
        "space"  : "O(N' param + T - N ' H + latent_dim)",
        "notes"  : "Learned adj used in both encoder and decoder graph prop.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (ELBO)",
        "trainable_graph_params": "N'",
    },
    "GCNForecaster_Learnable": {
        "time"   : "O(T - N' ' H + T - N ' H' + N')",
        "space"  : "O(N' param + T - N ' H)",
        "notes"  : "Same architecture as SpatioTemporalGNN2 but adj is learned, "
                   "output is N-dim forecast instead of binary.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (MSE forecast)",
        "trainable_graph_params": "N'",
    },
    "LearnableGraphForecaster": {
        "time"   : "O(T - N' ' H + T - N ' H' + N')",
        "space"  : "O(N' param + T - N ' H)",
        "notes"  : "DenseGCNConv-based; unsupervised forecast target.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (MSE forecast)",
        "trainable_graph_params": "N'",
    },
    "GATForecaster_Learnable": {
        "time"   : "O(B - E ' H - heads + H')",
        "space"  : "O(N' param + E - H ' heads)",
        "notes"  : "GAT + LSTM with learned edge gates; N-dim forecast output.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (MSE forecast)",
        "trainable_graph_params": "N'",
    },
}


# =============================================================================
# SECTION 8 - SCALING ANALYSIS
# =============================================================================

def run_scaling_analysis(model_registry: list, device: torch.device,
                          N_values: list = None,
                          T_values: list = None,
                          B: int = 1):
    """
    Measure how latency scales as N (sensors) or T (seq_len) increases.

    CRITICAL: A fresh edge_index must be built for EACH value of N because
    SpatioTemporalGNN2 uses edge_index at construction to precompute '....
    Reusing an edge_index built for N=10 at N=5 causes out-of-bounds indexing,
    and at N=50 wastes capacity. We build a synthetic sparse graph per N.

    Returns two DataFrames: one for N scaling, one for T scaling.
    """
    if N_values is None: N_values = [5, 10, 20, 50]
    if T_values is None: T_values = [12, 24, 48, 96]

    rows_N, rows_T = [], []

    for entry in model_registry:
        name = entry["name"]
        print(f"  Scaling: {name}")

        # '' Scale by N (fix T=24) '''''''''''''''''''''''''''''''''''''''''''''
        for N in N_values:
            try:
                # Build a fresh edge_index sized for THIS value of N
                ei = make_synthetic_edge_index(N, num_neighbors=min(3, N - 1))
                ei = ei.to(device)

                m, inp = entry["model_fn"](N, 24, ei)
                m = m.to(device).eval()
                inp = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                            for i in inp)

                with torch.no_grad():
                    for _ in range(5): m(*inp)   # warmup
                    if device.type == "cuda": torch.cuda.synchronize()
                    t0 = time.perf_counter()
                    for _ in range(20): m(*inp)
                    if device.type == "cuda": torch.cuda.synchronize()
                    lat = (time.perf_counter() - t0) / 20 * 1000

                params = sum(p.numel() for p in m.parameters() if p.requires_grad)
                rows_N.append({"Model": name, "N": N, "T": 24,
                                "Latency(ms)": round(lat, 3),
                                "Params": _fmt_params(params)})
            except Exception as e:
                rows_N.append({"Model": name, "N": N, "T": 24,
                                "Latency(ms)": f"ERR: {str(e)[:35]}",
                                "Params": "'"})
            finally:
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

        # '' Scale by T (fix N=10) '''''''''''''''''''''''''''''''''''''''''''''
        N_fixed = 10
        ei_fixed = make_synthetic_edge_index(N_fixed, num_neighbors=3).to(device)

        for T in T_values:
            try:
                m, inp = entry["model_fn"](N_fixed, T, ei_fixed)
                m = m.to(device).eval()
                inp = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                            for i in inp)

                with torch.no_grad():
                    for _ in range(5): m(*inp)
                    if device.type == "cuda": torch.cuda.synchronize()
                    t0 = time.perf_counter()
                    for _ in range(20): m(*inp)
                    if device.type == "cuda": torch.cuda.synchronize()
                    lat = (time.perf_counter() - t0) / 20 * 1000

                params = sum(p.numel() for p in m.parameters() if p.requires_grad)
                rows_T.append({"Model": name, "N": N_fixed, "T": T,
                                "Latency(ms)": round(lat, 3),
                                "Params": _fmt_params(params)})
            except Exception as e:
                rows_T.append({"Model": name, "N": N_fixed, "T": T,
                                "Latency(ms)": f"ERR: {str(e)[:35]}",
                                "Params": "'"})
            finally:
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

    return pd.DataFrame(rows_N), pd.DataFrame(rows_T)


# =============================================================================
# SECTION 9 - PER-LAYER BREAKDOWN PRINTER
# =============================================================================

def print_layer_breakdown(model: nn.Module, model_name: str):
    """Print a table of parameter count per named layer."""
    info = count_parameters(model)
    rows = [(name, _fmt_params(n), n)
            for name, n in info["per_layer"].items()]
    rows.sort(key=lambda r: r[2], reverse=True)   # largest layers first

    print(f"\n  {'='*55}")
    print(f"  Layer breakdown: {model_name}")
    print(f"  {'='*55}")
    if TABULATE:
        print(tabulate(
            [(r[0], r[1]) for r in rows],
            headers=["Layer", "Parameters"],
            tablefmt="simple"
        ))
    else:
        for name, fmt, _ in rows:
            print(f"    {name:<45} {fmt:>10}")

    print(f"  {'='*55}")
    print(f"  Total trainable : {_fmt_params(info['total_trainable'])}")
    if info["total_buffer"] > 0:
        print(f"  Buffers (non-trainable, e.g. A_hat): "
              f"{_fmt_params(info['total_buffer'])}")


# =============================================================================
# SECTION 10 - MAIN ANALYSIS RUNNER
# =============================================================================

def build_model_registry(N: int, T: int, edge_index: torch.Tensor) -> list:
    """
    Registry of all models with factory functions.

    Graph structure per model:
      SpatioTemporalGNN2  '  FIXED graph from edge_index (precomputed '... buffer)
      All other GNN models '  LEARN their own graph structure via adj_logits
                             or edge_logits parameters

    Each entry:  { name, model_fn(N, T, edge_index) '  (model, dummy_inputs), ... }
    The edge_index argument is only used by SpatioTemporalGNN2; other factories
    accept it but ignore it, so the call signature is uniform.
    """
    registry = [
        {
            "name"    : "TemporalCNN",
            "model_fn": lambda n, t, _ei=None: (
                TemporalCNN(n, t, hidden_dim=32, num_levels=4),
                (torch.randn(1, t, n),)
            ),
            "category": "CNN",
            "paradigm": "Supervised",
        },
        {
            "name"    : "StackedLSTM",
            "model_fn": lambda n, t, _ei=None: (
                StackedLSTM(n, hidden_dim=64, num_layers=2),
                (torch.randn(1, t, n),)
            ),
            "category": "RNN",
            "paradigm": "Supervised",
        },
        {
            "name"    : "SensorVAE",
            "model_fn": lambda n, t, _ei=None: (
                SensorVAE(n, t, hidden_dim=64, latent_dim=16),
                (torch.randn(1, t, n),)
            ),
            "category": "VAE",
            "paradigm": "Unsupervised",
        },
    ]

    if GNN_AVAILABLE:
        registry += [
            # '' ONLY model using a fixed graph ''''''''''''''''''''''''''''''''
            {
                "name"    : "SpatioTemporalGNN2",
                "model_fn": lambda n, t, _ei: (
                    SpatioTemporalGNN2(n, _ei, gcn_hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (fixed)",
                "paradigm": "Supervised",
            },
            # '' All others learn their graph ''''''''''''''''''''''''''''''''''
            {
                "name"    : "GNN_LearnableGraph",
                "model_fn": lambda n, t, _ei=None: (
                    GNN_LearnableGraph(n, hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Supervised",
            },
            {
                "name"    : "GATLSTM_LearnableGraph",
                "model_fn": lambda n, t, _ei=None: (
                    GATLSTM_LearnableGraph(n, t, hidden_dim=16, heads=2),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Supervised",
            },
            {
                "name"    : "GraphAnomalyVAE_Learnable",
                "model_fn": lambda n, t, _ei=None: (
                    GraphAnomalyVAE_Learnable(n, t, gcn_hidden=16, rnn_hidden=32, latent_dim=16),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN+VAE (learned)",
                "paradigm": "Unsupervised",
            },
            {
                "name"    : "GCNForecaster_Learnable",
                "model_fn": lambda n, t, _ei=None: (
                    GCNForecaster_Learnable(n, gcn_hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Unsupervised",
            },
            {
                "name"    : "LearnableGraphForecaster",
                "model_fn": lambda n, t, _ei=None: (
                    LearnableGraphForecaster(n, hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Unsupervised",
            },
            {
                "name"    : "GATForecaster_Learnable",
                "model_fn": lambda n, t, _ei=None: (
                    GATForecaster_Learnable(n, t, hidden_dim=16, heads=2),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Unsupervised",
            },
        ]

    return registry


def run_complexity_analysis(num_sensors: int  = 10,
                             seq_len: int      = 24,
                             adj_csv: str      = None,
                             batch_size: int   = 1,
                             run_scaling: bool = True,
                             verbose_layers: bool = True):
    """
    Main entry point. Prints:
      1. Per-model summary table (params, FLOPs, latency, memory, throughput)
      2. Per-layer parameter breakdown (if verbose_layers=True)
      3. Theoretical complexity table
      4. Scaling analysis (latency vs N and vs T)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    N, T   = num_sensors, seq_len

    print(f"\n{'='*70}")
    print(f"  Computational Complexity Analysis - Sensor Anomaly Detection Models")
    print(f"  Device     : {device}")
    print(f"  num_sensors: {N}  |  seq_len: {T}  |  batch_size: {batch_size}")
    print(f"{'='*70}\n")

    # '' Load / build edge_index '''''''''''''''''''''''''''''''''''''''''''''''
    if adj_csv and Path(adj_csv).exists():
        edge_index, _ = load_edge_index(adj_csv, N)
    else:
        if adj_csv:
            print(f"[!]  {adj_csv} not found - using synthetic random graph.")
        edge_index = make_synthetic_edge_index(N)
        n_edges = edge_index.shape[1]
        print(f"   Synthetic graph: {N} sensors, {n_edges} directed edges")
    print()

    registry = build_model_registry(N, T, edge_index)

    # '' Summary table '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
    summary_rows = []
    for entry in registry:
        name = entry["name"]
        print(f"  Profiling: {name} ...")
        try:
            # All model_fn factories take (N, T, edge_index) uniformly.
            # Non-GNN and learned-graph models ignore edge_index.
            model, dummy_inputs = entry["model_fn"](N, T, edge_index)
            model.eval()

            param_info = count_parameters(model)
            total_p    = param_info["total_trainable"]
            buf_p      = param_info["total_buffer"]

            flops_str  = count_flops(model, dummy_inputs, N, T, B=1)

            perf = measure_latency_and_throughput(
                model, dummy_inputs, device,
                n_warmup=10, n_runs=50, batch_size=batch_size
            )

            mem = measure_memory(model, dummy_inputs, device)

            summary_rows.append({
                "Model"          : name,
                "Category"       : entry["category"],
                "Paradigm"       : entry["paradigm"],
                "Trainable Params": _fmt_params(total_p),
                "Buffer Params"  : _fmt_params(buf_p) if buf_p > 0 else "'",
                "FLOPs"          : flops_str,
                "Latency (ms)"   : perf["latency_ms"],
                f"Throughput\n(samp/s)" : perf["throughput"],
                f"Peak {mem['device']} Mem (MB)": mem["peak_mb"],
            })

            if verbose_layers:
                print_layer_breakdown(model, name)

        except Exception as e:
            summary_rows.append({
                "Model"          : name,
                "Category"       : entry["category"],
                "Paradigm"       : entry["paradigm"],
                "Trainable Params": f"ERROR: {str(e)[:35]}",
                "Buffer Params"  : "'",
                "FLOPs"          : "'",
                "Latency (ms)"   : "'",
                f"Throughput\n(samp/s)": "'",
                f"Peak CPU Mem (MB)": "'",
            })
        finally:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    # '' Print summary table '''''''''''''''''''''''''''''''''''''''''''''''''''
    print(f"\n\n{'='*70}")
    print("  SUMMARY: Parameters, FLOPs, Latency, Memory, Throughput")
    print(f"{'='*70}")
    headers = list(summary_rows[0].keys())
    rows    = [[r[h] for h in headers] for r in summary_rows]
    if TABULATE:
        print(tabulate(rows, headers=headers, tablefmt="fancy_grid"))
    else:
        w = {h: max(len(str(h)), max(len(str(r[h])) for r in summary_rows))
             for h in headers}
        print("  ".join(str(h).ljust(w[h]) for h in headers))
        print("  ".join("-" * w[h] for h in headers))
        for r in summary_rows:
            print("  ".join(str(r[h]).ljust(w[h]) for h in headers))

    # '' Theoretical complexity table ''''''''''''''''''''''''''''''''''''''''''
    print(f"\n\n{'='*70}")
    print("  THEORETICAL COMPLEXITY  (Big-O in N=sensors, T=seq_len, H=hidden_dim)")
    print(f"{'='*70}")
    tc_rows = []
    for model_name, info in THEORETICAL_COMPLEXITY.items():
        tc_rows.append([
            model_name.replace("\n", " "),
            info["time"],
            info["space"],
            info["paradigm"],
            "Fixed" if info["graph_aware"] and info["trainable_graph_params"] == 0
                else ("Learned" if info["graph_aware"] else "N/A"),
            str(info["trainable_graph_params"]),
            info["notes"],
        ])
    tc_headers = ["Model", "Time", "Space", "Paradigm",
                  "Graph Type", "Graph Params", "Notes"]
    if TABULATE:
        print(tabulate(tc_rows, headers=tc_headers, tablefmt="grid"))
    else:
        for row in tc_rows:
            print(f"  {row[0]:<30} | {row[1]:<35} | {row[2]:<25}")
            print(f"    |   {row[5]}")

    # '' Scaling analysis ''''''''''''''''''''''''''''''''''''''''''''''''''''''
    if run_scaling:
        print(f"\n\n{'='*70}")
        print("  SCALING ANALYSIS")
        print(f"{'='*70}")

        print("\n  == Latency vs N (num_sensors), T fixed=24 ==")
        df_N, df_T = run_scaling_analysis(
            registry, device,
            N_values=[5, 10, 20, 50] if N <= 50 else [10, 25, 50, 100],
            T_values=[12, 24, 48, 96],
        )

        pivot_N = df_N.pivot(index="Model", columns="N", values="Latency(ms)")
        print()
        if TABULATE:
            print(tabulate(pivot_N, headers="keys", tablefmt="simple",
                           floatfmt=".3f"))
        else:
            print(pivot_N.to_string())

        print("\n  == Latency vs T (seq_len), N fixed=10 ==")
        pivot_T = df_T.pivot(index="Model", columns="T", values="Latency(ms)")
        print()
        if TABULATE:
            print(tabulate(pivot_T, headers="keys", tablefmt="simple",
                           floatfmt=".3f"))
        else:
            print(pivot_T.to_string())

    print(f"\n\n{'='*70}")
    print("  NOTES")
    print(f"{'='*70}")
    print("""
FLOPs
'''''
' Reported as MACs - 2 (each multiply-accumulate = 2 FLOPs).
' Suffix '(est.)' means thop could not trace the model (PyG custom ops)
  and the value was computed analytically from Linear/GRU/LSTM layers only.
  GCN propagation (einsum / DenseGCNConv) is not counted in est. values.
' To get accurate FLOPs for GNN models, install thop and ensure the model
  is traceable, or use torch.profiler for operation-level breakdown.

Buffer Params
'''''''''''''
' ''' means no non-trainable buffers.
' GNN models with fixed graphs store precomputed  '... = D^{-'}(A+I)D^{-'}
  as a register_buffer. This is NOT updated by the optimizer but DOES
  consume memory and affects FLOPs (N'N matmul per time step).

Latency
'''''''
' Measured as the average of 50 forward passes after 10 warmup passes.
' Batch size = 1 by default. Increase --batch_size to measure throughput
  under realistic training/inference conditions.

Scaling Analysis
''''''''''''''''
' Learnable-graph GNNs (GNN_LearnableGraph, LearnableGraphForecaster) show
  O(N') scaling in both parameters and FLOPs because adj_logits is N'N.
' Fixed-graph GNNs also do N'N matmuls but those do not scale with
  trainable params - the buffer stays fixed after construction.
' GATLSTM / GATForecaster have a Python loop over the batch dimension
  (no vectorisation), so their latency scales linearly with batch size.
""")

    return summary_rows


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    import sys
    _in_jupyter = "ipykernel" in sys.modules

    if _in_jupyter:
        # '' Notebook / %run mode - edit paths and settings here ''''''''''''''
        run_complexity_analysis(
            num_sensors    = 10,
            seq_len        = 24,
            adj_csv        = "Data/adjacency_matrices/lab_fne_matrix_norm.csv",   # '  set your adjacency CSV path
            batch_size     = 1,
            run_scaling    = True,
            verbose_layers = True,
        )
    else:
        # '' CLI mode ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
        parser = argparse.ArgumentParser(
            description="Computational complexity analysis for sensor anomaly models"
        )
        parser.add_argument("--adj_csv",       type=str,            help="Path to adjacency matrix CSV")
        parser.add_argument("--num_sensors",   type=int, default=10, help="Number of sensors (graph nodes)")
        parser.add_argument("--seq_len",       type=int, default=24, help="Input sequence length (time steps)")
        parser.add_argument("--batch_size",    type=int, default=1,  help="Batch size for latency/throughput")
        parser.add_argument("--no_scaling",    action="store_true",  help="Skip scaling analysis (faster)")
        parser.add_argument("--no_layers",     action="store_true",  help="Skip per-layer breakdowns")
        args = parser.parse_args()

        run_complexity_analysis(
            num_sensors    = args.num_sensors,
            seq_len        = args.seq_len,
            adj_csv        = args.adj_csv,
            batch_size     = args.batch_size,
            run_scaling    = not args.no_scaling,
            verbose_layers = not args.no_layers,
        )


  Computational Complexity Analysis - Sensor Anomaly Detection Models
  Device     : cuda
  num_sensors: 10  |  seq_len: 24  |  batch_size: 1

   Adjacency: 10 sensors, 86 directed edges, density=95.6%

  Profiling: TemporalCNN ...

  Layer breakdown: TemporalCNN
Layer             Parameters
----------------  ------------
net.0.c2.weight   3.1 K
net.1.c1.weight   3.1 K
net.1.c2.weight   3.1 K
net.2.c1.weight   3.1 K
net.2.c2.weight   3.1 K
net.3.c1.weight   3.1 K
net.3.c2.weight   3.1 K
net.0.c1.weight   960
net.0.res.weight  320
net.0.c1.bias     32
net.0.c2.bias     32
net.0.n1.weight   32
net.0.n1.bias     32
net.0.n2.weight   32
net.0.n2.bias     32
net.0.res.bias    32
net.1.c1.bias     32
net.1.c2.bias     32
net.1.n1.weight   32
net.1.n1.bias     32
net.1.n2.weight   32
net.1.n2.bias     32
net.2.c1.bias     32
net.2.c2.bias     32
net.2.n1.weight   32
net.2.n1.bias     32
net.2.n2.weight   32
net.2.n2.bias     32
net.3.c1.bias     32
net.3.c2.bias     32
net.3.n1.weight   32
n

In [44]:
## Scaled complexity analysis.
"""
complexity_analysis.py
=======================
Computational complexity analysis for all sensor anomaly detection models.

Covers:
  1. Learnable parameter count  (per layer and total)
  2. FLOPs per forward pass      (via thop; analytical fallback per layer type)
  3. Theoretical complexity      (Big-O in N, T, H)
  4. Forward-pass latency        (CPU wall-clock; GPU if available)
  5. Peak memory usage           (CPU RSS; GPU VRAM if available)
  6. Throughput                  (samples / second)
  7. Scaling analysis            (how each metric grows as N and T increase)

Usage (Jupyter):
    from complexity_analysis import run_complexity_analysis
    run_complexity_analysis(num_sensors=10, seq_len=24, adj_csv="adjacency.csv")

Usage (CLI):
    python complexity_analysis.py --adj_csv adjacency.csv --num_sensors 10 --seq_len 24

Dependencies:
    pip install thop tabulate psutil torch torch-geometric pandas numpy
"""

import argparse
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# '' Optional imports ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
try:
    from thop import profile as thop_profile
    THOP = True
except ImportError:
    THOP = False
    print("[!]  thop not installed - FLOPs will use analytical estimates.\n"
          "   Install with: pip install thop")

try:
    import psutil
    PSUTIL = True
except ImportError:
    PSUTIL = False

try:
    from tabulate import tabulate
    TABULATE = True
except ImportError:
    TABULATE = False

try:
    from torch_geometric.nn import GATConv
    from torch_geometric.nn.dense import DenseGCNConv
    GNN_AVAILABLE = True
except ImportError:
    GNN_AVAILABLE = False
    print("[!]  torch_geometric not found - GNN models will be skipped.")


# =============================================================================
# SECTION 1 - MODEL DEFINITIONS
# (Minimal copies of models from anomaly_pipeline.py, kept self-contained)
# =============================================================================

# '' CNN '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1, **kw):
        pad = (kernel_size - 1) * dilation
        super().__init__(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation, **kw)
        self._pad = pad

    def forward(self, x):
        out = super().forward(x)
        return out[:, :, :-self._pad] if self._pad > 0 else out


class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.c1  = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.c2  = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.n1  = nn.BatchNorm1d(out_ch)
        self.n2  = nn.BatchNorm1d(out_ch)
        self.res = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.drop = nn.Dropout(0.1)

    def forward(self, x):
        out = self.drop(torch.relu(self.n1(self.c1(x))))
        out = self.drop(torch.relu(self.n2(self.c2(out))))
        return torch.relu(out + self.res(x))


class TemporalCNN(nn.Module):
    """
    Temporal CNN (TCN): stacked dilated causal convolutions.
    Input  : (B, T, N)
    Output : (B,)  ' binary logit
    """
    def __init__(self, num_sensors, seq_len, hidden_dim=32, num_levels=4, kernel_size=3):
        super().__init__()
        chs = [num_sensors] + [hidden_dim] * num_levels
        self.net = nn.Sequential(*[
            TCNBlock(chs[i], chs[i+1], kernel_size, 2**i)
            for i in range(num_levels)
        ])
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out = self.net(x.permute(0, 2, 1))[:, :, -1]
        return self.fc(out).squeeze(-1)


# '' LSTM ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
class StackedLSTM(nn.Module):
    """
    Stacked LSTM encoder + linear decoder.
    Input  : (B, T, N)
    Output : (B,)
    """
    def __init__(self, num_sensors, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(num_sensors, hidden_dim, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :])).squeeze(-1)


# '' VAE '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
class SensorVAE(nn.Module):
    """
    LSTM-based VAE.
    Input  : (B, T, N)
    Output : recon (B, T, N), mu (B, latent), logvar (B, latent)
    """
    def __init__(self, num_sensors, seq_len, hidden_dim=64, latent_dim=16, beta=1.0):
        super().__init__()
        self.beta    = beta
        self.seq_len = seq_len
        # Encoder
        self.enc_lstm   = nn.LSTM(num_sensors, hidden_dim, batch_first=True)
        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar  = nn.Linear(hidden_dim, latent_dim)
        # Decoder
        self.dec_fc     = nn.Linear(latent_dim, hidden_dim)
        self.dec_lstm   = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.dec_out    = nn.Linear(hidden_dim, num_sensors)

    def forward(self, x):
        _, (h, _) = self.enc_lstm(x)
        h  = h[-1]
        mu = self.fc_mu(h)
        lv = self.fc_logvar(h)
        z  = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        d  = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        o, _ = self.dec_lstm(d)
        return self.dec_out(o), mu, lv

    def anomaly_score(self, x):
        self.eval()
        with torch.no_grad():
            recon, _, _ = self.forward(x)
        return F.mse_loss(recon, x, reduction='none').mean(dim=(1, 2))


# '' GNN models ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
if GNN_AVAILABLE:

    class SpatioTemporalGNN2(nn.Module):
        """Fixed-graph GCN + GRU (precomputed normalised adjacency)."""
        def __init__(self, num_sensors, edge_index, gcn_hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors    = num_sensors
            self.gcn_hidden_dim = gcn_hidden_dim
            A     = torch.zeros(num_sensors, num_sensors)
            A[edge_index[0], edge_index[1]] = 1.0
            A     = A + torch.eye(num_sensors)
            deg   = A.sum(dim=1).clamp(min=1e-6)
            d_inv = deg.pow(-0.5)
            self.register_buffer("A_hat", d_inv.unsqueeze(1) * A * d_inv.unsqueeze(0))
            self.W        = nn.Linear(1, gcn_hidden_dim, bias=False)
            self.gcn_bias = nn.Parameter(torch.zeros(gcn_hidden_dim))
            self.rnn      = nn.GRU(gcn_hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc       = nn.Linear(rnn_hidden_dim, 1)

        def forward(self, x, edge_index=None):
            B, T, N = x.shape
            node_feats = self.W(x.unsqueeze(-1))
            gcn_out    = torch.relu(
                torch.einsum("ij,btjh->btih", self.A_hat, node_feats) + self.gcn_bias
            )
            rnn_input = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)
            _, h_n    = self.rnn(rnn_input)
            rnn_out   = h_n.squeeze(0).reshape(B, N, -1)
            return self.fc(rnn_out.mean(dim=1)).squeeze(-1)

    class GNN_LearnableGraph(nn.Module):
        """Learnable adjacency (dense GCN) + GRU."""
        def __init__(self, num_sensors, hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors = num_sensors
            self.adj_logits  = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gcn_layer   = DenseGCNConv(in_channels=1, out_channels=hidden_dim)
            self.rnn         = nn.GRU(hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc          = nn.Linear(rnn_hidden_dim, 1)

        def forward(self, x):
            B, T, N = x.shape
            adj       = torch.softmax(self.adj_logits, dim=1)
            adj       = adj * (1 - torch.eye(N, device=x.device))
            adj_batch = adj.unsqueeze(0).expand(B, -1, -1)
            gcn_outs  = [self.gcn_layer(x[:, t, :].unsqueeze(-1), adj_batch)
                         for t in range(T)]
            gcn_out   = torch.stack(gcn_outs, dim=1)
            rnn_input = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)
            _, h_n    = self.rnn(rnn_input)
            rnn_out   = h_n.squeeze(0).reshape(B, N, -1)
            return self.fc(rnn_out.mean(dim=1)).squeeze(-1)

    class GATLSTM_LearnableGraph(nn.Module):
        """
        GAT + LSTM with LEARNED edge structure (fully-connected candidate graph).
        The graph is an N'N learnable mask: every possible edge exists, but
        its weight is a sigmoid-gated learnable parameter. GAT attention then
        further refines edge weights per head.
        """
        def __init__(self, num_sensors, seq_len, hidden_dim=16, heads=2):
            super().__init__()
            self.num_sensors = num_sensors
            self.edge_logits = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gat  = GATConv(seq_len, hidden_dim, heads=heads,
                                concat=True, dropout=0.1)
            self.lstm = nn.LSTM(num_sensors * hidden_dim * heads,
                                hidden_dim, batch_first=True)
            self.fc   = nn.Linear(hidden_dim, 1)
            self.relu = nn.ReLU()
            self.drop = nn.Dropout(0.1)

        def _get_edge_index(self, device):
            # Gate edges with a learnable mask (top-k selection each forward pass).
            # Keep all edges above sigmoid threshold 0.5.
            mask = torch.sigmoid(self.edge_logits) > 0.5
            mask.fill_diagonal_(False)
            src, dst = torch.where(mask)
            if len(src) == 0:   # fallback: dense graph if all gated off
                src, dst = torch.where(~torch.eye(self.num_sensors, dtype=torch.bool, device=device))
            return torch.stack([src, dst], dim=0).to(device)

        def forward(self, x):
            B = x.shape[0]
            edge_index = self._get_edge_index(x.device)
            gat_outs = [self.drop(self.relu(self.gat(x[i].T, edge_index)))
                        for i in range(B)]
            gat_out  = torch.stack(gat_outs).reshape(B, 1, -1)
            out, _   = self.lstm(gat_out)
            return self.fc(out[:, -1, :]).squeeze(-1)

    class GCNForecaster_Learnable(nn.Module):
        """
        Unsupervised GCN forecaster with LEARNED adjacency matrix.
        Identical architecture to SpatioTemporalGNN2 but with adj_logits
        (N'N learnable params) instead of a precomputed buffer, and
        predicts the full N-dim next step instead of a binary logit.
        """
        def __init__(self, num_sensors, gcn_hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors    = num_sensors
            self.gcn_hidden_dim = gcn_hidden_dim
            self.adj_logits = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.W          = nn.Linear(1, gcn_hidden_dim, bias=False)
            self.gcn_bias   = nn.Parameter(torch.zeros(gcn_hidden_dim))
            self.rnn        = nn.GRU(gcn_hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc         = nn.Linear(rnn_hidden_dim, num_sensors)

        def _get_adj(self, device):
            adj = torch.softmax(self.adj_logits, dim=1)
            return adj * (1 - torch.eye(self.num_sensors, device=device))

        def forward(self, x):
            B, T, N = x.shape
            x_in       = x[:, :-1, :]
            adj        = self._get_adj(x.device)
            node_feats = self.W(x_in.unsqueeze(-1))
            gcn_out    = torch.relu(
                torch.einsum("ij,btjh->btih", adj, node_feats) + self.gcn_bias
            )
            rnn_in = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
            out, _ = self.rnn(rnn_in)
            last   = out[:, -1, :].reshape(B, N, -1).mean(dim=1)
            return self.fc(last)

    class LearnableGraphForecaster(nn.Module):
        """Unsupervised forecaster with learnable adjacency."""
        def __init__(self, num_sensors, hidden_dim=16, rnn_hidden_dim=32):
            super().__init__()
            self.num_sensors = num_sensors
            self.adj_logits  = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gcn_layer   = DenseGCNConv(in_channels=1, out_channels=hidden_dim)
            self.rnn         = nn.GRU(hidden_dim, rnn_hidden_dim, batch_first=True)
            self.fc          = nn.Linear(rnn_hidden_dim, num_sensors)

        def forward(self, x):
            B, T, N = x.shape
            x_in     = x[:, :-1, :]
            adj      = torch.softmax(self.adj_logits, dim=1)
            adj      = adj * (1 - torch.eye(N, device=x.device))
            adj_b    = adj.unsqueeze(0).expand(B, -1, -1)
            gcn_outs = [self.gcn_layer(x_in[:, t, :].unsqueeze(-1), adj_b)
                        for t in range(T - 1)]
            gcn_out  = torch.stack(gcn_outs, dim=1)
            rnn_in   = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
            out, _   = self.rnn(rnn_in)
            last     = out[:, -1, :].reshape(B, N, -1).mean(dim=1)
            return self.fc(last)

    class GATForecaster_Learnable(nn.Module):
        """
        Unsupervised GAT forecaster with LEARNED edge structure.
        Same gated-edge approach as GATLSTM_LearnableGraph.
        """
        def __init__(self, num_sensors, seq_len, hidden_dim=16, heads=2):
            super().__init__()
            self.num_sensors = num_sensors
            self.edge_logits = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.gat  = GATConv(seq_len - 1, hidden_dim, heads=heads,
                                concat=True, dropout=0.1)
            self.lstm = nn.LSTM(num_sensors * hidden_dim * heads,
                                hidden_dim, batch_first=True)
            self.fc   = nn.Linear(hidden_dim, num_sensors)
            self.relu = nn.ReLU()
            self.drop = nn.Dropout(0.1)

        def _get_edge_index(self, device):
            mask = torch.sigmoid(self.edge_logits) > 0.5
            mask.fill_diagonal_(False)
            src, dst = torch.where(mask)
            if len(src) == 0:
                src, dst = torch.where(~torch.eye(self.num_sensors, dtype=torch.bool, device=device))
            return torch.stack([src, dst], dim=0).to(device)

        def forward(self, x):
            B, T, N = x.shape
            x_in = x[:, :-1, :]
            edge_index = self._get_edge_index(x.device)
            gat_outs = [self.drop(self.relu(self.gat(x_in[i].T, edge_index)))
                        for i in range(B)]
            gat_out  = torch.stack(gat_outs).reshape(B, 1, -1)
            out, _   = self.lstm(gat_out)
            return self.fc(out[:, -1, :])

    class GraphAnomalyVAE_Learnable(nn.Module):
        """
        Graph-aware VAE with LEARNED adjacency matrix.
        The adjacency is softmax-normalised from adj_logits (N'N trainable).
        """
        def __init__(self, num_sensors, seq_len,
                     gcn_hidden=16, rnn_hidden=32, latent_dim=16, beta=1.0):
            super().__init__()
            self.num_sensors = num_sensors
            self.seq_len     = seq_len
            self.beta        = beta
            self.adj_logits  = nn.Parameter(torch.randn(num_sensors, num_sensors))
            self.enc_node_proj = nn.Linear(1, gcn_hidden, bias=False)
            self.dec_node_proj = nn.Linear(1, gcn_hidden, bias=False)
            self.enc_rnn   = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
            self.fc_mu     = nn.Linear(rnn_hidden, latent_dim)
            self.fc_logvar = nn.Linear(rnn_hidden, latent_dim)
            self.dec_fc    = nn.Linear(latent_dim, rnn_hidden)
            self.dec_rnn   = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
            self.dec_gcn_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
            self.dec_out   = nn.Linear(gcn_hidden, 1)

        def _get_adj(self, device):
            adj = torch.softmax(self.adj_logits, dim=1)
            return adj * (1 - torch.eye(self.num_sensors, device=device))

        def forward(self, x):
            B, T, N = x.shape
            adj       = self._get_adj(x.device)
            feats     = self.enc_node_proj(x.unsqueeze(-1))
            gcn_out   = torch.relu(torch.einsum("ij,btjh->btih", adj, feats))
            rnn_input = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)
            _, h_n    = self.enc_rnn(rnn_input)
            h         = h_n.squeeze(0).reshape(B, N, -1).mean(dim=1)
            mu, lv    = self.fc_mu(h), self.fc_logvar(h)
            z         = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
            d         = self.dec_fc(z).unsqueeze(1).repeat(1, T, 1)
            rnn_out, _ = self.dec_rnn(d)
            node_h    = self.dec_gcn_W(rnn_out).unsqueeze(2).repeat(1, 1, N, 1)
            prop      = torch.relu(torch.einsum("ij,btjh->btih", adj, node_h))
            return self.dec_out(prop).squeeze(-1), mu, lv

        def anomaly_score(self, x):
            self.eval()
            with torch.no_grad():
                recon, _, _ = self.forward(x)
            return F.mse_loss(recon, x, reduction='none').mean(dim=(1, 2))


# =============================================================================
# SECTION 2 - ADJACENCY MATRIX LOADER
# =============================================================================

def load_edge_index(adj_csv_path: str, num_sensors: int):
    """Load adjacency matrix CSV and return (edge_index, adj_matrix)."""
    df  = pd.read_csv(adj_csv_path, index_col=None)
    first_col_numeric = pd.to_numeric(df.iloc[:, 0], errors="coerce").notna().all()
    if not first_col_numeric:
        df = pd.read_csv(adj_csv_path, index_col=0)
        adj = df.values.astype(np.float32)
    else:
        adj = df.values.astype(np.float32)

    if adj.shape[0] != num_sensors:
        raise ValueError(
            f"Adjacency matrix is {adj.shape[0]}'{adj.shape[0]} but "
            f"num_sensors={num_sensors}. Adjust --num_sensors."
        )
    np.fill_diagonal(adj, 0)
    rows, cols = np.where(adj > 0)
    edge_index = torch.tensor(np.stack([rows, cols], axis=0), dtype=torch.long)
    n_edges    = edge_index.shape[1]
    density    = n_edges / max(num_sensors * (num_sensors - 1), 1)
    print(f"   Adjacency: {num_sensors} sensors, {n_edges} directed edges, "
          f"density={100*density:.1f}%")
    return edge_index, adj


def make_synthetic_edge_index(num_sensors: int, num_neighbors: int = 3):
    """Fallback: build a random sparse edge_index when no CSV is provided."""
    rng = np.random.default_rng(42)
    rows, cols = [], []
    for i in range(num_sensors):
        nbrs = rng.choice(
            [j for j in range(num_sensors) if j != i],
            size=min(num_neighbors, num_sensors - 1),
            replace=False
        )
        for j in nbrs:
            rows += [i, j]; cols += [j, i]
    ei = torch.tensor([rows, cols], dtype=torch.long)
    return torch.unique(ei, dim=1)


# =============================================================================
# SECTION 3 - PARAMETER COUNT (per layer and total)
# =============================================================================

def count_parameters(model: nn.Module) -> dict:
    """
    Return a dict with per-layer and total learnable parameter counts.
    Buffers (e.g. A_hat) are counted separately as non-trainable.
    """
    layer_params  = {}
    total_train   = 0
    total_buffer  = 0

    for name, param in model.named_parameters():
        n = param.numel()
        layer_params[name] = n
        total_train += n

    for name, buf in model.named_buffers():
        total_buffer += buf.numel()

    return {
        "per_layer"     : layer_params,
        "total_trainable" : total_train,
        "total_buffer"  : total_buffer,   # e.g. precomputed A_hat
    }


def _fmt_params(n: int) -> str:
    if n >= 1_000_000: return f"{n/1e6:.3f} M"
    if n >= 1_000:     return f"{n/1e3:.1f} K"
    return str(n)


# =============================================================================
# SECTION 4 - FLOPs (thop + analytical per-layer fallback)
# =============================================================================

def _analytical_flops(model: nn.Module, dummy_inputs: tuple,
                       N: int, T: int, B: int = 1) -> str:
    """
    Layer-by-layer FLOPs estimate (MACs - 2 = FLOPs).

    Covers:
      nn.Linear / nn.Conv1d / nn.GRU / nn.LSTM  (exact)
      GCN propagation via  '... @ X        (N' ' H per time step)
      DenseGCNConv  via  adj @ X @ W    (N' ' H  +  N - H_in - H_out per step)
      GAT attention               (E - H ' heads)
      softmax over adj_logits     (N' per forward pass)

    The GCN/GAT estimates are based on model attribute names:
      - Modules containing 'gcn' or an  A_hat  buffer  '  GCN propagation cost
      - Modules containing 'gat' or a GATConv instance '  GAT edge attention cost
      - Parameters named 'adj_logits' or 'edge_logits' '  softmax / sigmoid cost

    Returns a human-readable string with '(est.)' suffix.
    """
    total_macs = 0
    has_graph_prop = False

    # '' Standard layer costs ''''''''''''''''''''''''''''''''''''''''''''''''''
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            total_macs += B * module.out_features * module.in_features
        elif isinstance(module, nn.Conv1d):
            L_out = T
            total_macs += (B * module.out_channels * L_out
                           * (module.in_channels // module.groups)
                           * module.kernel_size[0])
        elif isinstance(module, nn.GRU):
            H, I, L = module.hidden_size, module.input_size, module.num_layers
            total_macs += B * T * L * 3 * (H * H + H * I)
        elif isinstance(module, nn.LSTM):
            H, I, L = module.hidden_size, module.input_size, module.num_layers
            total_macs += B * T * L * 4 * (H * H + H * I)

    # '' Graph-specific costs (GCN propagation '... @ X via einsum) '''''''''''''''
    # Detect by presence of A_hat buffer or adj_logits / edge_logits parameter
    gcn_hidden = None
    for pname, _ in model.named_parameters():
        if "gcn" in pname.lower() or pname.endswith("W.weight"):
            # Try to infer hidden dim from model attributes
            for mod in model.modules():
                if isinstance(mod, nn.Linear) and mod.in_features == 1:
                    gcn_hidden = mod.out_features
                    break
            break

    # Detect GNN models: they have either A_hat buffer OR adj_logits parameter
    is_gnn = False
    for bname, _ in model.named_buffers():
        if "a_hat" in bname.lower():
            is_gnn = True
            break
    if not is_gnn:
        for pname, _ in model.named_parameters():
            if "adj_logits" in pname or "edge_logits" in pname:
                is_gnn = True
                break

    if is_gnn:
        # Estimate graph propagation cost.
        # '... @ X  where '... is (N,N) and X is (B, T, N, H) '  (B'T) matmuls of
        # size (N - N) ' (N - H) = N' ' H  per sample per time step.
        # Most models propagate over T steps (some over T-1); use T as estimate.
        if gcn_hidden is None:
            gcn_hidden = 16   # fallback based on our default config

        graph_prop_macs = B * T * N * N * gcn_hidden
        total_macs += graph_prop_macs

        # For decoder in GraphAnomalyVAE: a second graph propagation pass
        for mname, _ in model.named_modules():
            if "dec_rnn" in mname:
                total_macs += graph_prop_macs   # decoder also propagates
                break

        # Softmax over adj_logits (when learnable) contributes N' ops per forward
        for pname, _ in model.named_parameters():
            if "adj_logits" in pname or "edge_logits" in pname:
                total_macs += N * N * 3   # softmax: exp + sum + div
                break

        has_graph_prop = True

    # '' GATConv: attention over E edges '''''''''''''''''''''''''''''''''''''''
    if GNN_AVAILABLE:
        for mod in model.modules():
            if isinstance(mod, GATConv):
                H_in  = mod.in_channels
                H_out = mod.out_channels
                heads = mod.heads
                # Per sample: N nodes - H_in ' (H_out - heads) for linear transform
                # Plus ~E edges ' (H_out - heads) for attention
                # Approximate E as N' (dense) for the learnable-edge case
                E_approx = N * N
                total_macs += B * (N * H_in * H_out * heads
                                   + E_approx * H_out * heads)
                has_graph_prop = True

    flops = int(total_macs * 2)
    suffix = " (est.)"
    if not has_graph_prop:
        suffix = " (est., layers only)"

    if flops >= 1e9: return f"{flops/1e9:.3f} G{suffix}"
    if flops >= 1e6: return f"{flops/1e6:.2f} M{suffix}"
    if flops >= 1e3: return f"{flops/1e3:.1f} K{suffix}"
    return f"{flops}{suffix}"


def count_flops(model: nn.Module, dummy_inputs: tuple,
                N: int, T: int, B: int = 1) -> str:
    """
    Attempt thop profiling on CPU.
    Falls back to analytical estimate if thop fails or is unavailable.
    """
    if THOP:
        try:
            cpu_model  = model.to("cpu")
            cpu_inputs = tuple(i.to("cpu") if isinstance(i, torch.Tensor) else i
                               for i in dummy_inputs)
            macs, _ = thop_profile(cpu_model, inputs=cpu_inputs, verbose=False)
            flops = int(macs * 2)
            if flops >= 1e9: return f"{flops/1e9:.3f} G"
            if flops >= 1e6: return f"{flops/1e6:.2f} M"
            if flops >= 1e3: return f"{flops/1e3:.1f} K"
            return str(flops)
        except Exception:
            pass   # fall through to analytical

    return _analytical_flops(model, dummy_inputs, N, T, B)


# =============================================================================
# SECTION 5 - LATENCY & THROUGHPUT
# =============================================================================

def measure_latency_and_throughput(model: nn.Module, dummy_inputs: tuple,
                                   device: torch.device,
                                   n_warmup: int = 10,
                                   n_runs: int   = 50,
                                   batch_size: int = 1) -> dict:
    """
    Measure:
      latency_ms   : average single-forward-pass wall-clock time (ms)
      throughput   : samples per second
    """
    model = model.to(device).eval()
    inputs = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                   for i in dummy_inputs)

    with torch.no_grad():
        for _ in range(n_warmup):
            model(*inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(n_runs):
            model(*inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

    latency_ms = (elapsed / n_runs) * 1000
    throughput  = (batch_size * n_runs) / elapsed
    return {"latency_ms": round(latency_ms, 3),
            "throughput": round(throughput, 1)}


# =============================================================================
# SECTION 6 - MEMORY USAGE
# =============================================================================

def measure_memory(model: nn.Module, dummy_inputs: tuple,
                   device: torch.device) -> dict:
    """
    Measure peak memory for one forward pass.
      CPU: RSS delta via psutil (approximate)
      GPU: torch.cuda.max_memory_allocated
    """
    model = model.to(device).eval()
    inputs = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                   for i in dummy_inputs)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            model(*inputs)
        peak_mb = torch.cuda.max_memory_allocated(device) / 1024**2
        return {"device": "GPU", "peak_mb": round(peak_mb, 2)}

    elif PSUTIL:
        proc = psutil.Process()
        mem_before = proc.memory_info().rss / 1024**2
        with torch.no_grad():
            model(*inputs)
        mem_after = proc.memory_info().rss / 1024**2
        delta = max(mem_after - mem_before, 0.0)
        return {"device": "CPU", "peak_mb": round(delta, 2)}

    else:
        return {"device": "CPU", "peak_mb": "N/A (install psutil)"}


# =============================================================================
# SECTION 7 - THEORETICAL COMPLEXITY TABLE
# =============================================================================

THEORETICAL_COMPLEXITY = {
    "TemporalCNN": {
        "time"   : "O(T - C ' K - L)",
        "space"  : "O(T - C)",
        "notes"  : "C=hidden channels, K=kernel size, L=levels. "
                   "Fully parallel across time.",
        "graph_aware": False,
        "paradigm"   : "Supervised",
        "trainable_graph_params": 0,
    },
    "StackedLSTM": {
        "time"   : "O(T - H' ' L)",
        "space"  : "O(T - H)",
        "notes"  : "H=hidden dim, L=num_layers. Sequential time steps.",
        "graph_aware": False,
        "paradigm"   : "Supervised",
        "trainable_graph_params": 0,
    },
    "SensorVAE": {
        "time"   : "O(T - H') ' 2  (encoder + decoder)",
        "space"  : "O(T - H + latent_dim)",
        "notes"  : "LSTM encoder + LSTM decoder.",
        "graph_aware": False,
        "paradigm"   : "Unsupervised (ELBO)",
        "trainable_graph_params": 0,
    },
    "SpatioTemporalGNN2\n(ONLY fixed-graph model)": {
        "time"   : "O(T - N' ' H + T - N ' H')",
        "space"  : "O(N' buffer + T - N ' H)",
        "notes"  : "'... precomputed from edge_index, stored as non-trainable buffer. "
                   "No graph parameters to learn - graph structure is given.",
        "graph_aware": True,
        "paradigm"   : "Supervised",
        "trainable_graph_params": 0,
    },
    "GNN_LearnableGraph\n(learned adjacency)": {
        "time"   : "O(T - N' ' H + T - N ' H' + N')",
        "space"  : "O(N' param + T - N ' H)",
        "notes"  : "N' learnable adj_logits + softmax cost per forward pass.",
        "graph_aware": True,
        "paradigm"   : "Supervised",
        "trainable_graph_params": "N'",
    },
    "GATLSTM_LearnableGraph\n(learned edges)": {
        "time"   : "O(B - E ' H - heads + H')",
        "space"  : "O(N' param + E - H ' heads)",
        "notes"  : "N' edge_logits (sigmoid-gated) + GAT attention over active edges.",
        "graph_aware": True,
        "paradigm"   : "Supervised",
        "trainable_graph_params": "N'",
    },
    "GraphAnomalyVAE_Learnable": {
        "time"   : "O(T - N' ' H + T - N ' H') ' 2",
        "space"  : "O(N' param + T - N ' H + latent_dim)",
        "notes"  : "Learned adj used in both encoder and decoder graph prop.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (ELBO)",
        "trainable_graph_params": "N'",
    },
    "GCNForecaster_Learnable": {
        "time"   : "O(T - N' ' H + T - N ' H' + N')",
        "space"  : "O(N' param + T - N ' H)",
        "notes"  : "Same architecture as SpatioTemporalGNN2 but adj is learned, "
                   "output is N-dim forecast instead of binary.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (MSE forecast)",
        "trainable_graph_params": "N'",
    },
    "LearnableGraphForecaster": {
        "time"   : "O(T - N' ' H + T - N ' H' + N')",
        "space"  : "O(N' param + T - N ' H)",
        "notes"  : "DenseGCNConv-based; unsupervised forecast target.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (MSE forecast)",
        "trainable_graph_params": "N'",
    },
    "GATForecaster_Learnable": {
        "time"   : "O(B - E ' H - heads + H')",
        "space"  : "O(N' param + E - H ' heads)",
        "notes"  : "GAT + LSTM with learned edge gates; N-dim forecast output.",
        "graph_aware": True,
        "paradigm"   : "Unsupervised (MSE forecast)",
        "trainable_graph_params": "N'",
    },
}


# =============================================================================
# SECTION 8 - SCALING ANALYSIS
# =============================================================================

def run_scaling_analysis(model_registry: list, device: torch.device,
                          N_values: list = None,
                          T_values: list = None,
                          B: int = 1):
    """
    Measure how latency, parameters, and FLOPs scale as N (sensors) or T (seq_len) increases.

    CRITICAL: A fresh edge_index must be built for EACH value of N because
    SpatioTemporalGNN2 uses edge_index at construction to precompute '....
    Reusing an edge_index built for N=10 at N=5 causes out-of-bounds indexing,
    and at N=50 wastes capacity. We build a synthetic sparse graph per N.

    Returns two DataFrames: one for N scaling, one for T scaling.
    """
    if N_values is None: N_values = [5, 10, 20, 50]
    if T_values is None: T_values = [12, 24, 48, 96]

    rows_N, rows_T = [], []

    for entry in model_registry:
        name = entry["name"]
        print(f"  Scaling: {name}")

        # '' Scale by N (fix T=24) '''''''''''''''''''''''''''''''''''''''''''''
        for N in N_values:
            try:
                # Build a fresh edge_index sized for THIS value of N
                ei = make_synthetic_edge_index(N, num_neighbors=min(3, N - 1))
                ei = ei.to(device)

                m, inp = entry["model_fn"](N, 24, ei)
                m = m.to(device).eval()
                inp = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                            for i in inp)

                with torch.no_grad():
                    for _ in range(5): m(*inp)   # warmup
                    if device.type == "cuda": torch.cuda.synchronize()
                    t0 = time.perf_counter()
                    for _ in range(20): m(*inp)
                    if device.type == "cuda": torch.cuda.synchronize()
                    lat = (time.perf_counter() - t0) / 20 * 1000

                params = sum(p.numel() for p in m.parameters() if p.requires_grad)
                flops_str = count_flops(m, inp, N, 24, B=1)
                
                rows_N.append({"Model": name, "N": N, "T": 24,
                                "Latency(ms)": round(lat, 3),
                                "Params": params,
                                "Params_fmt": _fmt_params(params),
                                "FLOPs": flops_str})
            except Exception as e:
                rows_N.append({"Model": name, "N": N, "T": 24,
                                "Latency(ms)": f"ERR: {str(e)[:35]}",
                                "Params": 0,
                                "Params_fmt": "'",
                                "FLOPs": "'"})
            finally:
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

        # '' Scale by T (fix N=10) '''''''''''''''''''''''''''''''''''''''''''''
        N_fixed = 10
        ei_fixed = make_synthetic_edge_index(N_fixed, num_neighbors=3).to(device)

        for T in T_values:
            try:
                m, inp = entry["model_fn"](N_fixed, T, ei_fixed)
                m = m.to(device).eval()
                inp = tuple(i.to(device) if isinstance(i, torch.Tensor) else i
                            for i in inp)

                with torch.no_grad():
                    for _ in range(5): m(*inp)
                    if device.type == "cuda": torch.cuda.synchronize()
                    t0 = time.perf_counter()
                    for _ in range(20): m(*inp)
                    if device.type == "cuda": torch.cuda.synchronize()
                    lat = (time.perf_counter() - t0) / 20 * 1000

                params = sum(p.numel() for p in m.parameters() if p.requires_grad)
                flops_str = count_flops(m, inp, N_fixed, T, B=1)
                
                rows_T.append({"Model": name, "N": N_fixed, "T": T,
                                "Latency(ms)": round(lat, 3),
                                "Params": params,
                                "Params_fmt": _fmt_params(params),
                                "FLOPs": flops_str})
            except Exception as e:
                rows_T.append({"Model": name, "N": N_fixed, "T": T,
                                "Latency(ms)": f"ERR: {str(e)[:35]}",
                                "Params": 0,
                                "Params_fmt": "'",
                                "FLOPs": "'"})
            finally:
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

    return pd.DataFrame(rows_N), pd.DataFrame(rows_T)


# =============================================================================
# SECTION 9 - PER-LAYER BREAKDOWN PRINTER
# =============================================================================

def print_layer_breakdown(model: nn.Module, model_name: str):
    """Print a table of parameter count per named layer."""
    info = count_parameters(model)
    rows = [(name, _fmt_params(n), n)
            for name, n in info["per_layer"].items()]
    rows.sort(key=lambda r: r[2], reverse=True)   # largest layers first

    print(f"\n  {'='*55}")
    print(f"  Layer breakdown: {model_name}")
    print(f"  {'='*55}")
    if TABULATE:
        print(tabulate(
            [(r[0], r[1]) for r in rows],
            headers=["Layer", "Parameters"],
            tablefmt="simple"
        ))
    else:
        for name, fmt, _ in rows:
            print(f"    {name:<45} {fmt:>10}")

    print(f"  {'='*55}")
    print(f"  Total trainable : {_fmt_params(info['total_trainable'])}")
    if info["total_buffer"] > 0:
        print(f"  Buffers (non-trainable, e.g. A_hat): "
              f"{_fmt_params(info['total_buffer'])}")


# =============================================================================
# SECTION 10 - MAIN ANALYSIS RUNNER
# =============================================================================

def build_model_registry(N: int, T: int, edge_index: torch.Tensor) -> list:
    """
    Registry of all models with factory functions.

    Graph structure per model:
      SpatioTemporalGNN2  '  FIXED graph from edge_index (precomputed '... buffer)
      All other GNN models '  LEARN their own graph structure via adj_logits
                             or edge_logits parameters

    Each entry:  { name, model_fn(N, T, edge_index) '  (model, dummy_inputs), ... }
    The edge_index argument is only used by SpatioTemporalGNN2; other factories
    accept it but ignore it, so the call signature is uniform.
    """
    registry = [
        {
            "name"    : "TemporalCNN",
            "model_fn": lambda n, t, _ei=None: (
                TemporalCNN(n, t, hidden_dim=32, num_levels=4),
                (torch.randn(1, t, n),)
            ),
            "category": "CNN",
            "paradigm": "Supervised",
        },
        {
            "name"    : "StackedLSTM",
            "model_fn": lambda n, t, _ei=None: (
                StackedLSTM(n, hidden_dim=64, num_layers=2),
                (torch.randn(1, t, n),)
            ),
            "category": "RNN",
            "paradigm": "Supervised",
        },
        {
            "name"    : "SensorVAE",
            "model_fn": lambda n, t, _ei=None: (
                SensorVAE(n, t, hidden_dim=64, latent_dim=16),
                (torch.randn(1, t, n),)
            ),
            "category": "VAE",
            "paradigm": "Unsupervised",
        },
    ]

    if GNN_AVAILABLE:
        registry += [
            # '' ONLY model using a fixed graph ''''''''''''''''''''''''''''''''
            {
                "name"    : "SpatioTemporalGNN2",
                "model_fn": lambda n, t, _ei: (
                    SpatioTemporalGNN2(n, _ei, gcn_hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (fixed)",
                "paradigm": "Supervised",
            },
            # '' All others learn their graph ''''''''''''''''''''''''''''''''''
            {
                "name"    : "GNN_LearnableGraph",
                "model_fn": lambda n, t, _ei=None: (
                    GNN_LearnableGraph(n, hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Supervised",
            },
            {
                "name"    : "GATLSTM_LearnableGraph",
                "model_fn": lambda n, t, _ei=None: (
                    GATLSTM_LearnableGraph(n, t, hidden_dim=16, heads=2),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Supervised",
            },
            {
                "name"    : "GraphAnomalyVAE_Learnable",
                "model_fn": lambda n, t, _ei=None: (
                    GraphAnomalyVAE_Learnable(n, t, gcn_hidden=16, rnn_hidden=32, latent_dim=16),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN+VAE (learned)",
                "paradigm": "Unsupervised",
            },
            {
                "name"    : "GCNForecaster_Learnable",
                "model_fn": lambda n, t, _ei=None: (
                    GCNForecaster_Learnable(n, gcn_hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Unsupervised",
            },
            {
                "name"    : "LearnableGraphForecaster",
                "model_fn": lambda n, t, _ei=None: (
                    LearnableGraphForecaster(n, hidden_dim=16, rnn_hidden_dim=32),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Unsupervised",
            },
            {
                "name"    : "GATForecaster_Learnable",
                "model_fn": lambda n, t, _ei=None: (
                    GATForecaster_Learnable(n, t, hidden_dim=16, heads=2),
                    (torch.randn(1, t, n),)
                ),
                "category": "GNN (learned)",
                "paradigm": "Unsupervised",
            },
        ]

    return registry


def run_complexity_analysis(num_sensors: int  = 10,
                             seq_len: int      = 24,
                             adj_csv: str      = None,
                             batch_size: int   = 1,
                             run_scaling: bool = True,
                             verbose_layers: bool = True):
    """
    Main entry point. Prints:
      1. Per-model summary table (params, FLOPs, latency, memory, throughput)
      2. Per-layer parameter breakdown (if verbose_layers=True)
      3. Theoretical complexity table
      4. Scaling analysis (latency, params, FLOPs vs N and vs T)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    N, T   = num_sensors, seq_len

    print(f"\n{'='*70}")
    print(f"  Computational Complexity Analysis - Sensor Anomaly Detection Models")
    print(f"  Device     : {device}")
    print(f"  num_sensors: {N}  |  seq_len: {T}  |  batch_size: {batch_size}")
    print(f"{'='*70}\n")

    # '' Load / build edge_index '''''''''''''''''''''''''''''''''''''''''''''''
    if adj_csv and Path(adj_csv).exists():
        edge_index, _ = load_edge_index(adj_csv, N)
    else:
        if adj_csv:
            print(f"[!]  {adj_csv} not found - using synthetic random graph.")
        edge_index = make_synthetic_edge_index(N)
        n_edges = edge_index.shape[1]
        print(f"   Synthetic graph: {N} sensors, {n_edges} directed edges")
    print()

    registry = build_model_registry(N, T, edge_index)

    # '' Summary table '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
    summary_rows = []
    for entry in registry:
        name = entry["name"]
        print(f"  Profiling: {name} ...")
        try:
            # All model_fn factories take (N, T, edge_index) uniformly.
            # Non-GNN and learned-graph models ignore edge_index.
            model, dummy_inputs = entry["model_fn"](N, T, edge_index)
            model.eval()

            param_info = count_parameters(model)
            total_p    = param_info["total_trainable"]
            buf_p      = param_info["total_buffer"]

            flops_str  = count_flops(model, dummy_inputs, N, T, B=1)

            perf = measure_latency_and_throughput(
                model, dummy_inputs, device,
                n_warmup=10, n_runs=50, batch_size=batch_size
            )

            mem = measure_memory(model, dummy_inputs, device)

            summary_rows.append({
                "Model"          : name,
                "Category"       : entry["category"],
                "Paradigm"       : entry["paradigm"],
                "Trainable Params": _fmt_params(total_p),
                "Buffer Params"  : _fmt_params(buf_p) if buf_p > 0 else "'",
                "FLOPs"          : flops_str,
                "Latency (ms)"   : perf["latency_ms"],
                f"Throughput\n(samp/s)" : perf["throughput"],
                f"Peak {mem['device']} Mem (MB)": mem["peak_mb"],
            })

            if verbose_layers:
                print_layer_breakdown(model, name)

        except Exception as e:
            summary_rows.append({
                "Model"          : name,
                "Category"       : entry["category"],
                "Paradigm"       : entry["paradigm"],
                "Trainable Params": f"ERROR: {str(e)[:35]}",
                "Buffer Params"  : "'",
                "FLOPs"          : "'",
                "Latency (ms)"   : "'",
                f"Throughput\n(samp/s)": "'",
                f"Peak CPU Mem (MB)": "'",
            })
        finally:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    # '' Print summary table '''''''''''''''''''''''''''''''''''''''''''''''''''
    print(f"\n\n{'='*70}")
    print("  SUMMARY: Parameters, FLOPs, Latency, Memory, Throughput")
    print(f"{'='*70}")
    headers = list(summary_rows[0].keys())
    rows    = [[r[h] for h in headers] for r in summary_rows]
    if TABULATE:
        print(tabulate(rows, headers=headers, tablefmt="fancy_grid"))
    else:
        w = {h: max(len(str(h)), max(len(str(r[h])) for r in summary_rows))
             for h in headers}
        print("  ".join(str(h).ljust(w[h]) for h in headers))
        print("  ".join("-" * w[h] for h in headers))
        for r in summary_rows:
            print("  ".join(str(r[h]).ljust(w[h]) for h in headers))

    # '' Theoretical complexity table ''''''''''''''''''''''''''''''''''''''''''
    print(f"\n\n{'='*70}")
    print("  THEORETICAL COMPLEXITY  (Big-O in N=sensors, T=seq_len, H=hidden_dim)")
    print(f"{'='*70}")
    tc_rows = []
    for model_name, info in THEORETICAL_COMPLEXITY.items():
        tc_rows.append([
            model_name.replace("\n", " "),
            info["time"],
            info["space"],
            info["paradigm"],
            "Fixed" if info["graph_aware"] and info["trainable_graph_params"] == 0
                else ("Learned" if info["graph_aware"] else "N/A"),
            str(info["trainable_graph_params"]),
            info["notes"],
        ])
    tc_headers = ["Model", "Time", "Space", "Paradigm",
                  "Graph Type", "Graph Params", "Notes"]
    if TABULATE:
        print(tabulate(tc_rows, headers=tc_headers, tablefmt="grid"))
    else:
        for row in tc_rows:
            print(f"  {row[0]:<30} | {row[1]:<35} | {row[2]:<25}")
            print(f"    |   {row[5]}")

    # '' Scaling analysis ''''''''''''''''''''''''''''''''''''''''''''''''''''''
    if run_scaling:
        print(f"\n\n{'='*70}")
        print("  SCALING ANALYSIS")
        print(f"{'='*70}")

        print("\n  == Scaling by N (num_sensors), T fixed=24 ==")
        df_N, df_T = run_scaling_analysis(
            registry, device,
            N_values=[5, 10, 20, 50] if N <= 50 else [10, 25, 50, 100],
            T_values=[12, 24, 48, 96],
        )

        # Print comprehensive scaling tables
        print("\n  --- Latency (ms) vs N ---")
        pivot_N_lat = df_N.pivot(index="Model", columns="N", values="Latency(ms)")
        if TABULATE:
            print(tabulate(pivot_N_lat, headers="keys", tablefmt="simple",
                           floatfmt=".3f"))
        else:
            print(pivot_N_lat.to_string())

        print("\n  --- Parameters vs N ---")
        pivot_N_params = df_N.pivot(index="Model", columns="N", values="Params_fmt")
        if TABULATE:
            print(tabulate(pivot_N_params, headers="keys", tablefmt="simple"))
        else:
            print(pivot_N_params.to_string())

        print("\n  --- FLOPs vs N ---")
        pivot_N_flops = df_N.pivot(index="Model", columns="N", values="FLOPs")
        if TABULATE:
            print(tabulate(pivot_N_flops, headers="keys", tablefmt="simple"))
        else:
            print(pivot_N_flops.to_string())

        print("\n\n  == Scaling by T (seq_len), N fixed=10 ==")
        
        print("\n  --- Latency (ms) vs T ---")
        pivot_T_lat = df_T.pivot(index="Model", columns="T", values="Latency(ms)")
        if TABULATE:
            print(tabulate(pivot_T_lat, headers="keys", tablefmt="simple",
                           floatfmt=".3f"))
        else:
            print(pivot_T_lat.to_string())

        print("\n  --- Parameters vs T ---")
        pivot_T_params = df_T.pivot(index="Model", columns="T", values="Params_fmt")
        if TABULATE:
            print(tabulate(pivot_T_params, headers="keys", tablefmt="simple"))
        else:
            print(pivot_T_params.to_string())

        print("\n  --- FLOPs vs T ---")
        pivot_T_flops = df_T.pivot(index="Model", columns="T", values="FLOPs")
        if TABULATE:
            print(tabulate(pivot_T_flops, headers="keys", tablefmt="simple"))
        else:
            print(pivot_T_flops.to_string())

    print(f"\n\n{'='*70}")
    print("  NOTES")
    print(f"{'='*70}")
    print("""
FLOPs
'''''
' Reported as MACs - 2 (each multiply-accumulate = 2 FLOPs).
' Suffix '(est.)' means thop could not trace the model (PyG custom ops)
  and the value was computed analytically from Linear/GRU/LSTM layers only.
  GCN propagation (einsum / DenseGCNConv) is not counted in est. values.
' To get accurate FLOPs for GNN models, install thop and ensure the model
  is traceable, or use torch.profiler for operation-level breakdown.

Buffer Params
'''''''''''''
' ''' means no non-trainable buffers.
' GNN models with fixed graphs store precomputed  '... = D^{-'}(A+I)D^{-'}
  as a register_buffer. This is NOT updated by the optimizer but DOES
  consume memory and affects FLOPs (N'N matmul per time step).

Latency
'''''''
' Measured as the average of 50 forward passes after 10 warmup passes.
' Batch size = 1 by default. Increase --batch_size to measure throughput
  under realistic training/inference conditions.

Scaling Analysis
''''''''''''''''
' Learnable-graph GNNs (GNN_LearnableGraph, LearnableGraphForecaster) show
  O(N') scaling in both parameters and FLOPs because adj_logits is N'N.
' Fixed-graph GNNs also do N'N matmuls but those do not scale with
  trainable params - the buffer stays fixed after construction.
' GATLSTM / GATForecaster have a Python loop over the batch dimension
  (no vectorisation), so their latency scales linearly with batch size.
' Parameters that remain constant across N or T indicate model architecture
  components that are independent of input dimensions.
' FLOPs scale differently than parameters: linear layers scale with input size,
  while recurrent operations scale with sequence length.
""")

    return summary_rows


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    import sys
    _in_jupyter = "ipykernel" in sys.modules

    if _in_jupyter:
        # '' Notebook / %run mode - edit paths and settings here ''''''''''''''
        run_complexity_analysis(
            num_sensors    = 10,
            seq_len        = 24,
            adj_csv        = "Data/adjacency_matrices/lab_fne_matrix_norm.csv",   # '  set your adjacency CSV path
            batch_size     = 1,
            run_scaling    = True,
            verbose_layers = True,
        )
    else:
        # '' CLI mode ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
        parser = argparse.ArgumentParser(
            description="Computational complexity analysis for sensor anomaly models"
        )
        parser.add_argument("--adj_csv",       type=str,            help="Path to adjacency matrix CSV")
        parser.add_argument("--num_sensors",   type=int, default=10, help="Number of sensors (graph nodes)")
        parser.add_argument("--seq_len",       type=int, default=24, help="Input sequence length (time steps)")
        parser.add_argument("--batch_size",    type=int, default=1,  help="Batch size for latency/throughput")
        parser.add_argument("--no_scaling",    action="store_true",  help="Skip scaling analysis (faster)")
        parser.add_argument("--no_layers",     action="store_true",  help="Skip per-layer breakdowns")
        args = parser.parse_args()

        run_complexity_analysis(
            num_sensors    = args.num_sensors,
            seq_len        = args.seq_len,
            adj_csv        = args.adj_csv,
            batch_size     = args.batch_size,
            run_scaling    = not args.no_scaling,
            verbose_layers = not args.no_layers,
        )


  Computational Complexity Analysis - Sensor Anomaly Detection Models
  Device     : cuda
  num_sensors: 10  |  seq_len: 24  |  batch_size: 1

   Adjacency: 10 sensors, 86 directed edges, density=95.6%

  Profiling: TemporalCNN ...

  Layer breakdown: TemporalCNN
Layer             Parameters
----------------  ------------
net.0.c2.weight   3.1 K
net.1.c1.weight   3.1 K
net.1.c2.weight   3.1 K
net.2.c1.weight   3.1 K
net.2.c2.weight   3.1 K
net.3.c1.weight   3.1 K
net.3.c2.weight   3.1 K
net.0.c1.weight   960
net.0.res.weight  320
net.0.c1.bias     32
net.0.c2.bias     32
net.0.n1.weight   32
net.0.n1.bias     32
net.0.n2.weight   32
net.0.n2.bias     32
net.0.res.bias    32
net.1.c1.bias     32
net.1.c2.bias     32
net.1.n1.weight   32
net.1.n1.bias     32
net.1.n2.weight   32
net.1.n2.bias     32
net.2.c1.bias     32
net.2.c2.bias     32
net.2.n1.weight   32
net.2.n1.bias     32
net.2.n2.weight   32
net.2.n2.bias     32
net.3.c1.bias     32
net.3.c2.bias     32
net.3.n1.weight   32
n